<div align="center">

<img src="../images/Logo-Uni-Osnabrueck.jpg" width="300"/>

# Introduction to Computational Linguistics

</div>

## Regular expressions

### Cheatsheet

| Pattern | Meaning | Example match |
|---------|---------|---------------|
| `.` | Any character (except newline) | `c.t` → "cat", "cut", "c3t" |
| `*` | 0 or more of previous | `go*d` → "gd", "god", "good" |
| `+` | 1 or more of previous | `go+d` → "god", "good" (not "gd") |
| `?` | 0 or 1 of previous | `colou?r` → "color", "colour" |
| `^` | Start of string | `^Hello` → "Hello world" |
| `$` | End of string | `world$` → "Hello world" |
| `[abc]` | Any one of a, b, c | `[aeiou]` → any vowel |
| `[^abc]` | Any char NOT in set | `[^aeiou]` → any consonant |
| `\w` | Word character `[a-zA-Z0-9_]` | `\w+` → any word |
| `\d` | Digit `[0-9]` | `\d+` → "42", "2026" |
| `\s` | Whitespace | `\s+` → spaces, tabs |
| `\b` | Word boundary | `\bcat\b` → "cat" not "catch" |
| `(abc)` | Capture group | `(go)+` → "gogo" |
| `a\|b` | Either a or b | `cat\|dog` → "cat" or "dog" |
| `{n,m}` | Between n and m repetitions | `\d{2,4}` → "42", "2026" |

### Regex in Bash — `grep`

`grep` searches for lines matching a pattern.

```bash
grep 'pattern' file.txt        # basic search
grep -E 'pattern' file.txt     # extended regex (ERE)
grep -o 'pattern' file.txt     # print only the matched part
grep -i 'pattern' file.txt     # case-insensitive
```

In [ ]:
%%bash
TEXT="The cat sat on the mat. A cat in a hat. 2 cats, 3 dogs."

# Lines containing "cat"
echo "$TEXT" | grep -o 'cat'

# Words starting with capital letter
echo "$TEXT" | grep -oE '\b[A-Z][a-z]+'

# All numbers
echo "$TEXT" | grep -oE '\d+' || echo "$TEXT" | grep -oE '[0-9]+'

# Words ending in "at"
echo "$TEXT" | grep -oE '\b\w+at\b'

### Regex in Python — `re` module

The four functions you'll use most:

| Function | What it does |
|----------|-------------|
| `re.search(pattern, text)` | Find first match anywhere in text |
| `re.match(pattern, text)` | Match only at the **start** of text |
| `re.findall(pattern, text)` | Return **all** matches as a list |
| `re.sub(pattern, replacement, text)` | **Replace** matches |

In [ ]:
import re

text = "The cat sat on the mat. A cat in a hat. 2 cats, 3 dogs."

# 1) findall() returns a LIST of all matches
words_ending_at = re.findall(r'\b\w+at\b', text)
print("Words ending in 'at':", words_ending_at)

# 2) Find all numbers
numbers = re.findall(r'\d+', text)
print("Numbers found:", numbers)

# 3) search() finds only the FIRST match
match = re.search(r'\bcat\b', text)
print("First 'cat' found at position:", match.start())

# 4) sub() replaces every match with a new string
censored = re.sub(r'\bcat\b', '***', text)
print("Censored:", censored)

# 5) Parentheses () create groups — each group is captured separately
date_text = "Published on 2026-05-12"
dates = re.findall(r'(\d{4})-(\d{2})-(\d{2})', date_text)
print("Date parts (year, month, day):", dates)

## Word tokenization

### Space-based tokenization

Splitting on whitespace is the simplest approach — but it's naive. Punctuation sticks to words, contractions break unpredictably, and you get noisy tokens like `"father's"`, `"weep!"`, `"God,"`.

In [ ]:
import re

# Read the whole Shakespeare text into one big string
with open('../resources/shakespeare.txt', 'r') as f:
    raw = f.read()

# --- Method 1: split on whitespace (naive) ---
# split() with no argument cuts on any whitespace (spaces, tabs, newlines)
naive_tokens = raw.split()
print("Naive split - total tokens:", len(naive_tokens))
print("Sample:", naive_tokens[300:308])

# Problem: punctuation stays attached to words ("father's", "weep!", "God,")
# isalpha() is True only if the token contains letters and nothing else
noisy = []
for token in naive_tokens:
    if not token.isalpha():
        noisy.append(token)

print("\nTokens with punctuation/digits attached:", len(noisy))
print("Examples:", noisy[10:18])

# --- Method 2: regex tokenizer (better) ---
# [a-zA-Z]+ means "one or more letters in a row"
# This grabs only the letter parts — punctuation is left behind
word_tokens = re.findall(r"[a-zA-Z]+", raw)
print("\nRegex tokenizer - total tokens:", len(word_tokens))
print("Sample:", word_tokens[300:308])

# --- Build the vocabulary: the set of unique words (lowercased) ---
# A set() automatically removes duplicates
vocab = set()
for token in word_tokens:
    vocab.add(token.lower())

print("\nVocabulary size (unique words):", len(vocab))

### Simple Tokenization in UNIX

#### The first step: tokenizing

`tr -sc 'A-Za-z' '\n'` — keep only letters, replace everything else with a newline. One word per line.

In [ ]:
%%bash
tr -sc 'A-Za-z' '\n' < ../resources/shakespeare.txt | head -15

#### The second step: sorting

Pipe into `sort` — brings identical words together so we can count them.

In [ ]:
%%bash
tr -sc 'A-Za-z' '\n' < ../resources/shakespeare.txt | sort | head -15

#### More counting

Add `uniq -c` to count consecutive duplicates, then `sort -rn` to rank by frequency.

In [ ]:
%%bash
tr -sc 'A-Za-z' '\n' < ../resources/shakespeare.txt \
  | tr 'A-Z' 'a-z' \
  | sort \
  | uniq -c \
  | sort -rn \
  | head -20

### Tokenization in languages without spaces

In Chinese, Japanese, and Thai, words are **not separated by spaces**. Splitting on whitespace gives you nothing useful — you get full sentences as single "tokens". Segmentation requires dedicated algorithms or dictionaries.

| Language | Sentence | Space-split tokens |
|----------|----------|--------------------|
| English | `the cat sat` | `["the", "cat", "sat"]` ✓ |
| Chinese | `猫坐在垫子上` | `["猫坐在垫子上"]` ✗ |

In [3]:
en = "The cat sat on the mat"
zh = "猫坐在垫子上"          # "The cat sat on the mat" in Chinese

print("English — split():", en.split())
print(f"  → {len(en.split())} tokens\n")

print("Chinese — split():", zh.split())
print(f"  → {len(zh.split())} tokens  (whole sentence = one 'token'!)")
print()

# Chinese characters are already word-like units — iterate over chars as a proxy
print("Chinese — character-level (rough baseline):", list(zh))
print(f"  → {len(zh)} characters")

English — split(): ['The', 'cat', 'sat', 'on', 'the', 'mat']
  → 6 tokens

Chinese — split(): ['猫坐在垫子上']
  → 1 tokens  (whole sentence = one 'token'!)

Chinese — character-level (rough baseline): ['猫', '坐', '在', '垫', '子', '上']
  → 6 characters


## Byte Pair Encoding

Instead of splitting on spaces or individual characters, **BPE lets the data decide** how to tokenize. The result is *subword* tokens — whole words when they're frequent, character chunks when they're not.

| Approach | "newer" | "lower" (unseen) |
|----------|---------|-----------------|
| Whitespace | `newer` | `lower` ✗ OOV |
| Char-level | `n e w e r` | `l o w e r` |
| **BPE** | `newer_` | `low` + `er_` ✓ |

Two parts:
- **Token learner** — scans a training corpus and learns a vocabulary of merge rules
- **Token segmenter** — applies those rules (greedily, in learned order) to new text

### BPE token learner algorithm

In [ ]:
# Corpus from the slides
corpus = "low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new"

# --- Step 1: Count how often each word appears ---
word_counts = {}
for word in corpus.split():
    if word not in word_counts:
        word_counts[word] = 0
    word_counts[word] += 1

# --- Step 2: Build the initial character vocabulary ---
# Split each word into individual characters and add '_' as end-of-word marker
# Example: "low" → "l o w _"
vocab = {}
for word, count in word_counts.items():
    chars = list(word) + ['_']     # ['l', 'o', 'w', '_']
    key = ' '.join(chars)          # "l o w _"
    vocab[key] = count

print("Initial character vocabulary:")
for word, count in vocab.items():
    print(" ", count, "x", word)


# --- Helper 1: count how often each adjacent pair of symbols appears ---
def count_pairs(vocab):
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()                  # ["l", "o", "w", "_"]
        # Walk through neighbors: (symbols[0], symbols[1]), (symbols[1], symbols[2]), ...
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            if pair not in pairs:
                pairs[pair] = 0
            pairs[pair] += freq
    return pairs


# --- Helper 2: find the pair with the highest count ---
def find_best_pair(pairs):
    best_pair = None
    best_count = 0
    for pair, count in pairs.items():
        if count > best_count:
            best_count = count
            best_pair = pair
    return best_pair


# --- Helper 3: merge the best pair everywhere in the vocab ---
def merge_pair(best_pair, vocab):
    new_vocab = {}
    old = best_pair[0] + ' ' + best_pair[1]    # "e r"
    new = best_pair[0] + best_pair[1]          # "er"
    for word, count in vocab.items():
        new_word = word.replace(old, new)      # replace every occurrence
        new_vocab[new_word] = count
    return new_vocab


# --- Run BPE: learn k merges ---
k = 10
merges = []    # we store the merge rules here for the segmenter

print("\nLearning merges:")
for i in range(k):
    pairs = count_pairs(vocab)
    best_pair = find_best_pair(pairs)
    vocab = merge_pair(best_pair, vocab)
    merges.append(best_pair)
    merged = best_pair[0] + best_pair[1]
    print("Merge", i + 1, ":", best_pair[0], "+", best_pair[1], "→", merged,
          " (count =", pairs[best_pair], ")")

print("\nFinal vocabulary:")
for word, count in vocab.items():
    print(" ", count, "x", word)

### BPE token segmenter algorithm

Apply the learned merges **greedily and in order** to any new word. Training frequencies don't matter here — only the merge sequence does.

- `newer` → seen in training → becomes one token `newer_`
- `lower` → unseen → partially matches → splits into `low` + `er_`

In [ ]:
def segment(word, merges):
    # Start by splitting the word into characters + end marker
    # "newer" → ['n', 'e', 'w', 'e', 'r', '_']
    symbols = list(word) + ['_']

    # Apply each learned merge rule, in the order it was learned
    for pair in merges:
        merged = pair[0] + pair[1]   # e.g. "e" + "r" → "er"

        # Walk through the symbols and build a new list,
        # merging neighbors whenever they match this rule.
        new_symbols = []
        i = 0
        while i < len(symbols):
            # If this symbol + the next one match the rule, glue them together
            if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i + 1] == pair[1]:
                new_symbols.append(merged)
                i += 2    # skip both symbols, they've been merged
            else:
                new_symbols.append(symbols[i])
                i += 1

        symbols = new_symbols

    return symbols


# Test on words from the slides
test_words = ["newer", "lower", "lowest", "widest", "newish"]

print("Word          Tokens")
print("-" * 40)
for word in test_words:
    tokens = segment(word, merges)
    print(word.ljust(14), tokens)

## Word Normalization and other issues

### Word Normalization

Converting tokens into a **standard form** before processing. Common tasks:

| Task | Before | After |
|------|--------|-------|
| Lowercase | `Fed` | `fed` |
| Remove abbrev. dots | `U.S.A.` | `USA` |
| Collapse repeated chars | `loooove` | `loove` |
| Normalize numbers | `$4.99` | keep or strip |

In [ ]:
import re

# --- 1) Remove periods from abbreviations ---
# str.replace(old, new) just swaps every "." with "" (empty string)
abbrevs = ["U.S.A.", "Ph.D.", "m.p.h.", "e.g."]
print("Abbreviation normalization:")
for w in abbrevs:
    print(" ", w, "→", w.replace('.', ''))

# --- 2) Collapse repeated characters like "loooove" → "loove" ---
# The pattern (.)\1{2,} reads as:
#   (.)     → capture ANY single character into group 1
#   \1      → "the SAME character as group 1" (called a back-reference)
#   {2,}    → 2 or more times in a row
# So this matches a character followed by at least 2 more copies of itself.
# We replace it with \1\1 (the same character, twice).
text = "I loooooove this! Sooooo gooood!"
normalized = re.sub(r'(.)\1{2,}', r'\1\1', text)
print("\nCollapse repeated chars:")
print("  Before:", text)
print("  After: ", normalized)

### Case folding

In [ ]:
sentence = "The FBI investigated General Motors and Apple Inc."

# Lowercasing helps search: "Apple" and "apple" match the same query
print("Original :", sentence)
print("Lowercased:", sentence.lower())

# But case sometimes carries meaning — lowercasing loses that
print()
pairs = [
    ("US",      "country name"),
    ("us",      "pronoun"),
    ("Apple",   "the company"),
    ("apple",   "the fruit"),
    ("General", "military rank"),
    ("general", "adjective"),
]
for word, meaning in pairs:
    print(" ", word.ljust(10), "=", meaning, " → lower:", word.lower())

### Lemmatization

In [ ]:
# Lemmatization maps words to their dictionary base form (the "lemma")
# am / is / are / was / were  →  be
# cats / cat's              →  cat
# running / ran             →  run

# A small manual lemma dictionary
lemma_dict = {
    "am":       "be",
    "is":       "be",
    "are":      "be",
    "was":      "be",
    "were":     "be",
    "cats":     "cat",
    "running":  "run",
    "ran":      "run",
    "better":   "good",
    "best":     "good",
    "studies":  "study",
}

sentence = "The cats are running and she was better than before"

words      = sentence.split()
lemmatized = []
for word in words:
    lower_word = word.lower()
    # If we know the lemma, use it — otherwise keep the word as-is
    lemma = lemma_dict.get(lower_word, lower_word)
    lemmatized.append(lemma)

print("Original :", sentence)
print("Lemmatized:", ' '.join(lemmatized))
print()
print("Note: real lemmatizers (like spaCy) use large dictionaries")
print("and part-of-speech info to handle thousands of word forms.")

### Stemming

In [ ]:
# Stemming: crudely chop off suffixes — faster but less accurate than lemmatization
# It doesn't use a dictionary, just rules.

def simple_stem(word):
    word = word.lower()
    # Rules are checked in order — the first match wins
    if word.endswith("tion"):
        return word[:-4]
    if word.endswith("ness"):
        return word[:-4]
    if word.endswith("ing"):
        return word[:-3]
    if word.endswith("est"):
        return word[:-3]
    if word.endswith("ed"):
        return word[:-2]
    if word.endswith("ly"):
        return word[:-2]
    if word.endswith("er"):
        return word[:-2]
    if word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    return word

test_words = ["running", "cats", "happily", "fastest", "computed", "connection"]

print("Word            Stem")
print("-" * 30)
for word in test_words:
    print(word.ljust(16), simple_stem(word))

### Porter Stemmer

In [ ]:
# The Porter Stemmer runs rules in a cascade (output of step 1 feeds into step 2, etc.)
# Here is Step 1a — just the plural/verb suffix rules:

def porter_step1a(word):
    word = word.lower()

    if word.endswith("sses"):   # caresses → caress
        return word[:-2]
    if word.endswith("ies"):    # ponies → poni
        return word[:-2]
    if word.endswith("ss"):     # caress → caress (leave alone)
        return word
    if word.endswith("s"):      # cats → cat
        return word[:-1]
    return word

examples = ["caresses", "ponies", "caress", "cats", "buses", "happiness"]

print("Word            After Step 1a")
print("-" * 35)
for word in examples:
    print(word.ljust(16), porter_step1a(word))

print()
print("The full Porter Stemmer has 5 steps and ~60 rules.")
print("Python's NLTK library has a complete built-in implementation:")

### Sentence Segmentation

In [ ]:
import re

text = "Dr. Smith works at Apple Inc. in Palo Alto. She earns $2.5M per year. Is that a lot? Yes!"

# --- Naive approach: split on every . ! or ? ---
naive = re.split(r'[.!?]', text)
print("Naive split (wrong):")
for s in naive:
    s = s.strip()
    if s:
        print(" ", repr(s))

# Problem: "Dr.", "Inc.", and "2.5" all trigger a split — false positives!


# --- Better approach: ignore periods after known abbreviations ---
# A set of words that often end with a period but DON'T end a sentence
abbreviations = {"dr", "mr", "mrs", "ms", "prof", "inc", "ltd", "vs", "etc"}


def split_sentences(text):
    sentences = []
    current = ""

    # Go word by word, adding each one to the current sentence
    for word in text.split():
        current = current + word + " "

        # Does this word end with sentence-ending punctuation?
        last_char = word[-1]
        if last_char not in '.!?':
            continue   # not the end of a sentence, keep going

        # Strip punctuation off the word and check if it's an abbreviation.
        # "Dr." → "dr"  →  found in abbreviations → don't split here.
        clean = word.rstrip('.!?').lower()
        if clean in abbreviations:
            continue   # it's an abbreviation, keep building the sentence

        # Otherwise: end of sentence! Save it and start a new one.
        sentences.append(current.strip())
        current = ""

    # Don't forget the last sentence if it didn't end with punctuation
    if current.strip():
        sentences.append(current.strip())

    return sentences


print("\nBetter split:")
for s in split_sentences(text):
    print(" ", repr(s))